In [6]:
import time
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score)

P = "../data/processed/"
X, y = {}, {}
for k in ["train", "validation", "test"]:
    X[k] = pd.read_parquet(f"{P}X_{k}.parquet")
    y[k] = pd.read_parquet(f"{P}y_{k}.parquet")["y"]

def best_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return thr[np.argmax(f1[:-1])]

def scores(y_true, p, thr):
    pred = (p >= thr).astype(int)
    return {"threshold": round(float(thr), 3), "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred), "f1": f1_score(y_true, pred),
            "pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p)}

results = {}
def run(name, model):
    t0 = time.time()
    model.fit(X["train"], y["train"])
    p_val = model.predict_proba(X["validation"])[:, 1]
    thr = best_threshold(y["validation"], p_val)          # threshold chosen on validation
    results[name] = {"model": model, "thr": thr, "val": scores(y["validation"], p_val, thr)}
    print(f"{name:22s} done in {time.time() - t0:.0f}s | {results[name]['val']}")

### Step 1 – Loading the data and common evaluation

We load the three saved sets (same files for the whole team). All models are trained on **train**, compared on **validation**, and the **test** set is used only once at the end.

Metrics: precision, recall and F1 of the high-severity class (`y = 1`), plus PR-AUC and ROC-AUC. Accuracy alone would be misleading because 34 % of the accidents are positive.

By default a model predicts "1" when its probability is above 0.5, which is not the best choice for an unbalanced target. For each model we therefore pick the threshold that maximizes F1 **on validation**, and we apply this same threshold on test. PR-AUC and ROC-AUC do not depend on the threshold.

In [7]:
run("Baseline (prior)", DummyClassifier(strategy="prior"))
run("Logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))

Baseline (prior)       done in 0s | {'threshold': 0.338, 'precision': 0.3359440674143481, 'recall': 1.0, 'f1': 0.5029313361367752, 'pr_auc': 0.3359440674143481, 'roc_auc': 0.5}
Logistic regression    done in 5s | {'threshold': 0.253, 'precision': 0.46573533424283764, 'recall': 0.8681314210151562, 'f1': 0.6062366813467822, 'pr_auc': 0.5778342495026013, 'roc_auc': 0.7509228512392554}


### Step 2 – Baseline and logistic regression

- **Baseline:** a model that ignores the features and gives every accident the same probability. With the F1-optimal threshold it ends up predicting "1" for everyone: recall 1.0, precision about 0.34, F1 about 0.50. Any real model must beat this.
- **Logistic regression:** simple, fast and interpretable, so it is our reference model. The numeric columns are standardized first (scaler learned on train), because linear models are sensitive to the scale of the features. `max_iter=1000` so that the solver converges.

Result on validation: the baseline reaches F1 = 0.503 (precision 0.336, recall 1.0, PR-AUC 0.336). The logistic regression reaches F1 = 0.606, PR-AUC = 0.578 and ROC-AUC = 0.751. It beats the baseline, but its precision (0.466) is only a bit above the 0.336 of the baseline: it finds most high-severity accidents (recall 0.868) at the price of many false alarms. This suggests that the link between the features and the target is not linear.

In [8]:
run("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=200, early_stopping=False, random_state=42))

HistGradientBoosting   done in 47s | {'threshold': 0.354, 'precision': 0.6145240095851616, 'recall': 0.8034381039568711, 'f1': 0.696396610856896, 'pr_auc': 0.7425950081452113, 'roc_auc': 0.8583478982612331}


### Step 3 – Gradient boosting

HistGradientBoosting builds 200 small trees one after another, each correcting the errors of the previous ones. It usually gives the best results on tabular data, needs no scaling, and is fast on 488k rows. Default hyperparameters for now (no tuning yet), `random_state=42` for reproducibility.

Result on validation: F1 = 0.696, precision 0.615, recall 0.803, PR-AUC = 0.743, ROC-AUC = 0.858 (threshold 0.354). Compared with the logistic regression, F1 gains +0.09 and PR-AUC +0.17, which confirms that non-linear effects and interactions (for example location × hour × weather) matter. Training takes 47 s.

In [9]:
run("Random forest", RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=42))

Random forest          done in 69s | {'threshold': 0.36, 'precision': 0.5983852682538223, 'recall': 0.7946088902451429, 'f1': 0.6826765942199967, 'pr_auc': 0.7280773470493818, 'roc_auc': 0.8451524742249655}


### Step 4 – Random forest

100 independent trees trained on random samples, whose votes are averaged. It is a second tree-based model to compare with boosting. `min_samples_leaf=5` avoids very deep trees that memorize the training data; `n_jobs=-1` uses all CPU cores.

Result on validation: F1 = 0.683, precision 0.598, recall 0.795, PR-AUC = 0.728, ROC-AUC = 0.845 (threshold 0.360). It is clearly better than the logistic regression and slightly below the gradient boosting.

In [10]:
print(pd.DataFrame({n: r["val"] for n, r in results.items()}).T.round(3))

test = {n: scores(y["test"], r["model"].predict_proba(X["test"])[:, 1], r["thr"]) for n, r in results.items()}
print(pd.DataFrame(test).T.round(3))

                      threshold  precision  recall     f1  pr_auc  roc_auc
Baseline (prior)          0.338      0.336   1.000  0.503   0.336    0.500
Logistic regression       0.253      0.466   0.868  0.606   0.578    0.751
HistGradientBoosting      0.354      0.615   0.803  0.696   0.743    0.858
Random forest             0.360      0.598   0.795  0.683   0.728    0.845
                      threshold  precision  recall     f1  pr_auc  roc_auc
Baseline (prior)          0.338      0.339   1.000  0.507   0.339    0.500
Logistic regression       0.253      0.463   0.858  0.601   0.580    0.746
HistGradientBoosting      0.354      0.613   0.787  0.689   0.728    0.849
Random forest             0.360      0.594   0.775  0.673   0.710    0.833


### Step 5 – Comparison

First table: validation (used to choose the threshold and compare models). Second table: test, computed once, with the thresholds fixed on validation.

Best model: | Model (test set) | Precision | Recall | F1 | PR-AUC | ROC-AUC |
|---|---|---|---|---|---|
| Baseline | 0.339 | 1.000 | 0.507 | 0.339 | 0.500 |
| Logistic regression | 0.463 | 0.858 | 0.601 | 0.580 | 0.746 |
| HistGradientBoosting | 0.613 | 0.787 | **0.689** | **0.728** | **0.849** |
| Random forest | 0.594 | 0.775 | 0.673 | 0.710 | 0.833 |

**Best model: HistGradientBoosting** (F1 = 0.689, PR-AUC = 0.728 on test). Its precision (0.613) is about 1.8 times the share of positives (0.339), so it is much better than a random guess, but there are still many errors.

Observations:
- **Validation → test:** F1 drops by only 0.007 for the boosting, 0.010 for the forest and 0.005 for the logistic regression. The thresholds chosen on validation transfer well to a later year, and there is no sign of overfitting to the validation set.
- **Tree-based models vs linear:** both beat the logistic regression by about 0.08 to 0.09 F1.
- **Boosting vs forest:** the gap is small (0.016 F1), and no hyperparameter was tuned yet.

Next steps: tune the hyperparameters of the best model on validation (learning rate, number of iterations, tree depth), look at the most important features, and analyse errors by state and by hour.